# 03 · Query a learned flow

**Goal:** load the PACE model trained in Notebook 02, query its velocity field, and integrate an ODE. Run Notebook 02 first, or replace `RUN_DIR` with one of your own training outputs.

Load only checkpoints you created or trust: Lightning checkpoints contain Python metadata. For biological data, query points must be transformed using the training-only scaler before inference.

In [1]:
from pathlib import Path
import os, sys
ROOT = Path.cwd().resolve()
if not (ROOT / 'train.py').exists():
    ROOT = ROOT.parent
assert (ROOT / 'train.py').exists(), 'Open this notebook from PACE/ or PACE/notebooks/.'
os.chdir(ROOT)
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'
import torch
torch.set_num_threads(1)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image
plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False})
print('PACE workspace:', ROOT.name)


PACE workspace: PACE


In [2]:
from src.inference import load_run, rollout
from src.methods.mfm.runner import build_datamodule
RUN_DIR = ROOT/'results/notebook_demo/results/oceans_positions_dim2_test1-3-5-7/pace'
if not (RUN_DIR/'resolved_config.yaml').exists():
    raise FileNotFoundError('Run Notebook 02 first, or set RUN_DIR to your own PACE run.')
model, cfg, checkpoint = load_run(RUN_DIR)
dm = build_datamodule(cfg)
dm.setup()
print('Loaded:', checkpoint.name)
print('Training labels:', list(dm.unique_train_labels))
print('Held-out labels:', list(dm.unique_test_labels))

Loaded: flow-epoch=001.ckpt
Training labels: [0, 2, 4, 6, 8]
Held-out labels: [1, 3, 5, 7]


## Integrate from the first snapshot

Training time is normalized using the training labels. Ocean labels 0 through 8 correspond to model time 0 through 1. This is an **end-to-end** rollout: unlike the paper's per-holdout metric, it does not restart from each observed bracket. A short tutorial run need not recover the vortex.

In [3]:
source = dm.train_frames[min(dm.unique_train_labels)]
trajectory = rollout(model, source, t_start=0., t_end=1., steps=101)
print('Trajectory shape [time, particles, coordinates]:', tuple(trajectory.shape))
fig, ax = plt.subplots(figsize=(7, 5))
for points in {**dm.train_frames, **dm.test_frames}.values():
    ax.scatter(*points.numpy().T, s=8, color='#b8c6d3', alpha=.35)
for j in range(0, source.shape[0], 2):
    ax.plot(trajectory[:,j,0], trajectory[:,j,1], color='#168b9d', alpha=.6, lw=1)
ax.scatter(*trajectory[0].T, c='#102b4a', s=20, label='Initial cloud')
ax.scatter(*trajectory[-1].T, c='#e33845', s=20, label='Predicted final cloud')
ax.set_title('PACE · short tutorial rollout'); ax.legend(); plt.show()

Trajectory shape [time, particles, coordinates]: (101, 24, 2)


## Query the velocity field directly

A model call takes `time` with shape `[N, 1]` and states with shape `[N, D]`.
The output is a velocity vector for each state. The same interface works for MFM.

In [4]:
with torch.no_grad():
    t = torch.full((len(source), 1), .25)
    velocity = model(t, source)
assert velocity.shape == source.shape
assert torch.isfinite(velocity).all()
display(pd.DataFrame({'x': source[:,0], 'y': source[:,1], 'vx': velocity[:,0], 'vy': velocity[:,1]}).head())

,x,y,vx,vy
0,-0.594702,-0.216803,-0.068331,-0.037076
1,-0.623898,-0.226699,-0.068484,-0.036834
2,-0.591002,-0.226329,-0.068416,-0.037162
3,-0.554049,-0.288553,-0.068964,-0.037778
4,-0.616941,-0.275575,-0.068858,-0.037126


## Recompute bracketed holdout metrics

This uses the same training-time scale and 101 Euler grid points as the training evaluator. Reference velocities are not needed to compute distribution metrics.

In [5]:
from src.methods.mfm.eval_metrics import evaluate_stage2_rollout_metrics
rows = evaluate_stage2_rollout_metrics(model, dm.selected_train_frames, dm.test_frames,
                                       list(dm.unique_train_labels), list(dm.unique_test_labels))
display(pd.DataFrame(rows)[['test_label','mmd','w1','w2']].round(4))

,test_label,mmd,w1,w2
0,1,1.0688,0.3626,0.3630
1,3,1.0961,0.4212,0.4226
2,5,1.0652,0.5625,0.5637
3,7,1.1176,0.6178,0.6198


## Make your own animation

After training a converged Ocean model, render a GIF from its actual trajectories:

```bash
python scripts/render_demo.py results/oceans_positions_dim2_test1-3-5-7/pace \
  --output results/my-ocean.gif
```

The renderer records a checkpoint hash, data hash, seed and epoch counts next to the GIF. See [the method guide](../docs/method.md) for the bridge-to-velocity equations.

## Reproduce the README reprogramming animation

The README uses WOT reprogramming data. Train one PACE model on a CUDA GPU, then render its three held-out intervals:

```bash
python train.py experiment=wot_fle_reprogramming_demo
python scripts/render_reprogramming_demo.py results/reprogramming_demo/results/wot_fle_three_interpolation_dim2_test6-11-16/pace
```

The panels show intervals 5→7, 10→12 and 15→17, crossing held-out snapshots 6, 11 and 16. Each starts from its observed training frame; all three use the same learned field and the original training-only scaler. They advance by the same interpolation fraction, not the same absolute time. These are population transport predictions, not observed individual cell lineages.

See [the biological demo notes](../docs/development-demo.md) for the complete configuration, diagnostics and an optional embryoid-body example. The earlier [Ocean tuning history](../docs/ocean-demo-tuning.md) remains available separately.

The demo preset uses Hungarian matching, sigma 0.02 and 1,000 velocity epochs,
selected using holdout feedback to reduce dispersion. It is separate from the paper preset;
see [the full refinement record](../docs/reprogramming-refinement.json).
